In [1]:
import pandas as pd
from pandas_datareader import data as web

In [2]:
from datetime import date

# Grab data from FRED
raw_pce = web.DataReader("PCEPILFE", "fred", '2023-01-01', date.today())
raw_wei = web.DataReader("WEI", "fred", '2023-01-01', date.today())

In [3]:
# Lag each indicator for publication lag
raw_pce["PCEPILFE"] = raw_pce["PCEPILFE"].shift(-1)
raw_wei["WEI"] = raw_wei["WEI"].shift(-1)

In [4]:
# Add average of last 3 observations to missing last value

average_change = round(((raw_pce["PCEPILFE"].iloc[-2] - raw_pce["PCEPILFE"].iloc[-3]) + 
    (raw_pce["PCEPILFE"].iloc[-3] - raw_pce["PCEPILFE"].iloc[-4]) + 
    (raw_pce["PCEPILFE"].iloc[-4] - raw_pce["PCEPILFE"].iloc[-5])) / 3 , 3)

raw_pce["PCEPILFE"].iloc[-1] = raw_pce["PCEPILFE"].iloc[-2] + average_change


average_change = round(((raw_wei["WEI"].iloc[-2] - raw_wei["WEI"].iloc[-3]) + 
    (raw_wei["WEI"].iloc[-3] - raw_wei["WEI"].iloc[-4]) + 
    (raw_wei["WEI"].iloc[-4] - raw_wei["WEI"].iloc[-5])) / 3, 3)

raw_wei["WEI"].iloc[-1] = raw_wei["WEI"].iloc[-2] + average_change

/var/folders/r8/f4vjftzx1s93lfzrk2s0hk9w0000gq/T/ipykernel_13206/157490068.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  raw_pce["PCEPILFE"].iloc[-1] = raw_pce["PCEPILFE"].iloc[-2] + average_change
/var/folders/r8/f4vjftzx1s93lfzrk2s0h

In [5]:
# PCE as % change from first date
anchor_date = pd.Timestamp("2024-07-01")
base_value = raw_pce.loc[anchor_date, "PCEPILFE"]
raw_pce["PCE"] = raw_pce["PCEPILFE"] / base_value

raw_pce = raw_pce.drop(columns= 'PCEPILFE')

In [6]:
# Read master dataframe
master = pd.read_parquet('./data/master.parquet')

# Get master date range and create a list of daily dates
start, end = master["Date"].min(), master["Date"].max()
daily_range = pd.date_range(start=start, end=end, freq="D")

#Forward fill FRED indices
daily_pce =  raw_pce["PCE"].reindex(daily_range).ffill()
daily_wei = raw_wei["WEI"].reindex(daily_range).ffill()

daily_pce["Date"] = daily_pce["DATE"]
daily_pce.drop(columns="DATE")

daily_wei["Date"] = daily_wei["DATE"]
daily_wei.drop(columns="DATE")

FileNotFoundError: [Errno 2] No such file or directory: './data/master.parquet'

In [ ]:
master = master.merge(daily_pce, on="Date", how="left")

In [ ]:
master = master.merge(daily_wei, on="Date", how="left")